# Full Ablation for the Safeguarded LLM-in-the-Loop ETL Paper

Runs the 2×2 ablation (**circular / cross-only / rules-only / hybrid**) **plus the Reflexion
baseline** on UCI Adult and KramaBench water-quality data, then emits the pooled results
table as paste-ready LaTeX.

**Before running:**
1. GPU runtime (Kaggle: T4/P100; Colab: T4 or better).
2. A Hugging Face token with access granted to `meta-llama/Llama-3.1-8B-Instruct` and
   `mistralai/Mistral-7B-Instruct-v0.2`.
   - Kaggle: add it as a Secret named `HF_TOKEN`.
   - Colab: add it via the key icon (Secrets) as `HF_TOKEN`.

Rough budget: models load once per mode; with 5 seeds expect a few hours on a T4.
Start with `SEEDS = "1,2,3"` if constrained — 3 seeds already beats the single-seed status quo.

In [ ]:
SEEDS = "1,2,3"   # 3 seeds is enough for the resubmission table; add more later if budget allows
REPO_ZIP_URL = "https://github.com/dhiaamejdi/LLM-in-the-Loop-Feedback-Driven-ETL/archive/refs/heads/main.zip"


In [ ]:
# Get the HF token from Kaggle or Colab secrets, or the environment
import os
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass
assert hf_token, "Set an HF_TOKEN secret (see instructions above)."
os.environ["HF_TOKEN"] = hf_token
print("Token loaded (not printed).")

In [ ]:
%pip -q install "transformers>=4.43" "accelerate>=0.30" "bitsandbytes>=0.43" "huggingface_hub>=0.23" pandas numpy
import torch; print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
if torch.cuda.is_available():
    gb = torch.cuda.get_device_properties(0).total_memory/1e9
    print(f"GPU memory: {gb:.1f} GB -> models will load in 4-bit automatically (fits worker + checker together)")

In [ ]:
# Fetch the code — ALWAYS refresh so a stale copy from a previous session is never used.
# (An earlier version of this cell skipped the download if the file existed, which made
#  Kaggle silently keep running old code after the repo was fixed.)
import urllib.request, zipfile, os, glob, shutil, hashlib

FORCE_REFRESH = True   # set False only if you deliberately edited files locally in /kaggle/working

if FORCE_REFRESH:
    for stale in ["repo.zip", "safeguarded_LLM_in_the_loop.py", "aggregate_results.py"]:
        if os.path.exists(stale):
            os.remove(stale)
    for d in glob.glob("LLM-in-the-Loop-*"):
        shutil.rmtree(d, ignore_errors=True)

if not os.path.exists("safeguarded_LLM_in_the_loop.py"):
    urllib.request.urlretrieve(REPO_ZIP_URL, "repo.zip")
    with zipfile.ZipFile("repo.zip") as z: z.extractall(".")
    root = glob.glob("LLM-in-the-Loop-*")[0]
    for f in ["safeguarded_LLM_in_the_loop.py", "aggregate_results.py"]:
        shutil.copy(os.path.join(root, f), ".")
    shutil.copytree(os.path.join(root, "datasets"), "datasets", dirs_exist_ok=True)

# Verify we truly have the fixed version before spending any GPU time:
src = open("safeguarded_LLM_in_the_loop.py").read()
assert "return_dict=True" in src and "attention_mask=attention_mask" in src, \
    "Stale script detected! The downloaded file lacks the generate() fix — check REPO_ZIP_URL / push the latest code."
print("Fetched fresh code. Script fingerprint:", hashlib.sha256(src.encode()).hexdigest()[:12])
print(os.listdir("."))

In [ ]:
# === Runner with per-batch progress in the notebook and automatic result zipping ===
import subprocess, sys, os, shutil, time
os.makedirs("runs", exist_ok=True)

def run_mode(tag, extra_args):
    log = f"runs/{tag}.log"
    print(f"\n########## {tag} ##########  (log: {log})", flush=True)
    t0 = time.time()
    with open(log, "w") as logf:
        p = subprocess.Popen([sys.executable, "-u", "safeguarded_LLM_in_the_loop.py"] + extra_args,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:                       # stream: progress is visible live
            logf.write(line)
            if any(k in line for k in ("batch", "Report saved", "Transform cache", "Evicting", "WARNING", "Error")):
                print("   ", line.rstrip(), flush=True)
        p.wait()
    mins = (time.time() - t0) / 60
    if p.returncode != 0:
        print(open(log).read()[-2500:])
        print(f"!! {tag} FAILED (exit {p.returncode}) after {mins:.1f} min — continuing.", flush=True)
    else:
        print(f"OK: {tag} in {mins:.1f} min", flush=True)
    # checkpoint results after EVERY mode so a session death never loses finished work
    shutil.make_archive("runs_checkpoint", "zip", "runs")
    return p.returncode

# === UCI Adult: 2x2 + Reflexion ===
# The worker transform is cached per seed and reused by all five modes automatically.
for mode in ["hybrid", "circular", "reflexion", "cross-only", "rules-only"]:
    run_mode(f"uci-{mode}", ["--mode", mode, "--seeds", SEEDS, "--outfile-prefix", "runs/uci"])

# Aggregate immediately: UCI-only table is already a usable resubmission result
subprocess.run([sys.executable, "aggregate_results.py", "runs"])
print(open("runs/ablation_table.tex").read())

In [ ]:
# === KramaBench water-quality: same matrix at a T4-feasible size ===
# 150 rows / 50 checked per seed. The full-scale (320k-row) evidence for this dataset
# already exists from the deterministic-path runs; these LLM-mode runs add the
# cross-architecture comparison on real-lake data without blowing the session budget.
for mode in ["hybrid", "circular", "reflexion", "cross-only", "rules-only"]:
    run_mode(f"kb-{mode}", ["--mode", mode, "--seeds", SEEDS,
                            "--data-csv", "datasets/water-body-testing-2019.csv",
                            "--rules-json", "datasets/kramabench_water_rules.json",
                            "--sample-size", "150", "--heldout-size", "50",
                            "--outfile-prefix", "runs/kb"])

In [ ]:
# === Aggregate: pooled stats + paste-ready LaTeX table ===
import subprocess, sys
subprocess.run([sys.executable, "aggregate_results.py", "runs"], check=True)
print(open("runs/ablation_table.tex").read())

## What to do with the output
1. **Download `runs_checkpoint.zip`** (right panel → Output) — it is refreshed after every mode,
   so even if the session dies you keep everything finished so far.
2. Paste `runs/ablation_table.tex` into **Section 7.3** of the paper.
3. Key comparisons: **circular vs. cross-only** (H2), **circular vs. reflexion** (does self-critique
   recover blind spots?), **rules-only vs. hybrid** (what the LLM adds).
4. Commit the `runs/` folder to the repo so every number is verifiable.

**If the session is killed mid-run:** re-running the notebook resumes cheaply — worker transforms
are cached per seed in the raw output files, so completed transform work is reused automatically
(only if `/kaggle/working` persisted; otherwise the matrix restarts, which is why the checkpoint
zip after every mode matters).

**Timing expectations (T4, 3 seeds):** the first mode of each dataset pays the worker cost
(UCI: 15 transform batches; KB: 24); every later mode reuses the cache and only runs its checker.
If a single worker batch takes more than ~3 minutes in the streamed output, stop and reduce
`--sample-size` rather than letting the session time out.